# Business Questions Analysis - Part 4

This notebook contains the analysis for the 5 business questions from Part 4 of the assignment.

## Business Questions:
1. What are the demographic differences between the top 13 performing and lowest 13 performing LGAs based on estimated revenue per active listing over the last 12 months?
2. Is there a correlation between the median age of a neighbourhood (from Census data) and the revenue generated per active listing in that neighbourhood?
3. What will be the best type of listing (property type, room type and accommodates) for the top 15 "listing_neighbourhood" (in terms of estimated revenue per active listing) to have the highest number of stays?
4. For hosts with multiple listings in Vic are their properties concentrated within the same LGA, or are they distributed across different LGAs?
5. For hosts with a single Airbnb listing does the estimated revenue over the last 12 months cover the annualised median mortgage repayment in the corresponding LGA? Which LGA has the highest percentage of hosts that can cover it?


In [32]:
# Import required libraries
import psycopg2
import pandas as pd
from sqlalchemy import create_engine
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")


Libraries imported successfully!


In [33]:
# Database connection parameters
DB_CONFIG = {
    'host': '34.40.238.227',
    'port': 5432,
    'database': 'postgres',
    'user': 'postgres',
    'password': '0t_:ETvs1.n|vMs,'
}

def get_connection():
    """Create a connection to PostgreSQL database"""
    try:
        conn = psycopg2.connect(**DB_CONFIG)
        return conn
    except Exception as e:
        print(f"Error connecting to database: {e}")
        return None

def get_sqlalchemy_engine():
    """Create SQLAlchemy engine for pandas operations"""
    try:
        connection_string = f"postgresql://{DB_CONFIG['user']}:{DB_CONFIG['password']}@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"
        engine = create_engine(connection_string)
        return engine
    except Exception as e:
        print(f"Error creating SQLAlchemy engine: {e}")
        return None

print("Database connection functions defined!")


Database connection functions defined!


In [34]:
# Test database connection and set up schema configuration
engine = get_sqlalchemy_engine()
if engine:
    print("✅ Database connection successful!")
    
    # Configure schema names
    SCHEMAS = {
        'bronze': 'public_bronze',
        'silver': 'public_silver', 
        'gold': 'public_gold',
        'snapshots': 'public_snapshots'
    }
    
    print("Using schemas:", SCHEMAS)
    
    # Verify schemas exist
    for name, schema in SCHEMAS.items():
        try:
            exists_df = pd.read_sql(
                f"SELECT COUNT(*) as n FROM information_schema.schemata WHERE schema_name = '{schema}'",
                engine
            )
            print(f"Schema {schema}: {'FOUND' if exists_df['n'].iloc[0] > 0 else 'MISSING'}")
        except Exception as e:
            print(f"Error checking schema {schema}: {e}")
else:
    print("❌ Database connection failed!")


✅ Database connection successful!
Using schemas: {'bronze': 'public_bronze', 'silver': 'public_silver', 'gold': 'public_gold', 'snapshots': 'public_snapshots'}
Schema public_bronze: FOUND
Schema public_silver: FOUND
Schema public_gold: FOUND
Schema public_snapshots: MISSING


In [35]:
# Check what tables exist in each schema
if engine:
    for layer in ['bronze', 'silver', 'gold', 'snapshots']:
        schema = SCHEMAS[layer]
        try:
            tables_df = pd.read_sql(
                f"""
                SELECT table_name
                FROM information_schema.tables
                WHERE table_schema = '{schema}'
                ORDER BY table_name
                """,
                engine
            )
            print(f"\nTables in {schema}:")
            if not tables_df.empty:
                print(tables_df['table_name'].tolist())
            else:
                print("No tables found")
        except Exception as e:
            print(f"Error listing tables in {schema}: {e}")



Tables in public_bronze:
['bronze_airbnb_listings', 'bronze_census_g01', 'bronze_census_g02', 'bronze_lga_mapping', 'bronze_nsw_lga_code']

Tables in public_silver:
['silver_airbnb_listings', 'silver_census_g01', 'silver_census_g02', 'silver_lga_mapping', 'silver_nsw_lga_code']

Tables in public_gold:
['dim_host', 'dim_lga', 'dim_neighbourhood', 'dim_property', 'dim_suburb', 'dm_host_neighbourhood', 'dm_listing_neighbourhood', 'dm_property_type', 'fact_listings']

Tables in public_snapshots:
No tables found


## Business Question 1: Demographic Differences Between Top and Bottom Performing LGAs

**Question:** What are the demographic differences (e.g., age group distribution, household size) between the top 13 performing and lowest 13 performing LGAs based on estimated revenue per active listing over the last 12 months?
